# Create and review clinical cases

**Goal:** Draft source-based cases, review them manually, then assemble the final databank.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Draft a case from its source

The study contains 20 adapted cases from each of six published casebooks. The reusable prompt below asks for a vignette, a task-specific reference and supporting pages. The supplied historical cases can be inspected without regenerating them. The historical assembly algorithm is in clinical_databank.py and expects the private source drafts and evidence audit.

In [ ]:
from sleepinn_study.workflow import clinical_authoring_prompt
packet = {"source_pdf": "licensed_casebook.pdf", "pages": [{"page": 1, "text": "Supply the selected case here."}]}
print(clinical_authoring_prompt(packet, "suspected_diagnosis"))
cases = read_jsonl(ROOT / "data/candidates/clinical.jsonl")
display(pd.DataFrame([{"source": c["source_pdf"], "task": c["metadata"]["question_task"]} for c in cases]).value_counts().rename("cases"))

## 3. Review clinical questions and references

Check that the question can be answered from the vignette, the reference addresses the requested task, and no patient findings were invented. This is the human dataset-review step, not model-answer scoring. Every case needs a recorded decision for a new final export.

In [ ]:
from sleepinn_study.review import QuestionReviewer
clinical_reviewer = QuestionReviewer(cases, ROOT / "outputs/human_review/clinical")
display(clinical_reviewer.widget)

## 4. Assemble the final database

After both review sessions finish, point to their exported JSONL files. The function verifies both completion manifests and hashes before combining the accepted records. The supplied release already contains the author-confirmed final knowledge and clinical banks.

In [ ]:
from sleepinn_study.workflow import assemble_final
ASSEMBLE_NEW_BANK = False
if ASSEMBLE_NEW_BANK:
    final_path = assemble_final(
        ROOT / "outputs/human_review/knowledge/exports/CHOOSE_REVIEW_EXPORT.jsonl",
        ROOT / "outputs/human_review/clinical/exports/CHOOSE_REVIEW_EXPORT.jsonl",
        ROOT / "outputs/final_databank/combined.jsonl")
    print(final_path)
else:
    print("Released final bank:", ROOT / "data/final")

## 5. Check the assembled release

Validation belongs after creation and review. Confirm unique IDs, valid question types and the expected final population.

In [ ]:
from sleepinn_study.databank import validate_items
final = read_jsonl(ROOT / "data/final/knowledge.jsonl") + read_jsonl(ROOT / "data/final/clinical.jsonl")
display(validate_items(final))
assert len(final) == 1335